In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

x_train = pd.read_csv(
    "x_train_final.csv",
    header=None,
    names=["idx", "idx2", "train", "gare", "date", "arret",
           "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"],
    skiprows=1
)
y_train = pd.read_csv("y_train_final_j5KGWWK.csv")

print(f"x_train : {x_train.shape[0]:,} lignes")

# =============================================================================
# 2. FEATURES
# =============================================================================

FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

# =============================================================================
# 3. SPLIT 90 / 10
# =============================================================================

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

print(f"Train      : {X_tr.shape[0]:,} lignes")
print(f"Validation : {X_val.shape[0]:,} lignes")

# =============================================================================
# 4. RANDOM FOREST
# =============================================================================

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

# =============================================================================
# 5. ÉVALUATION
# =============================================================================

y_pred = rf.predict(X_val)

print(f"\nMAE  : {mean_absolute_error(y_val, y_pred):.4f}")

x_train : 667,264 lignes
Train      : 600,537 lignes
Validation : 66,727 lignes


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   45.7s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.0min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.7s



MAE  : 0.8741


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.4s finished


In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# =============================================================================
# 1. CHARGEMENT
# =============================================================================

x_train = pd.read_csv(
    "x_train_final.csv",
    header=None,
    names=["idx", "idx2", "train", "gare", "date", "arret",
           "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"],
    skiprows=1
)
y_train = pd.read_csv("y_train_final_j5KGWWK.csv")
x_test  = pd.read_csv("x_test_final.csv")

# =============================================================================
# 2. FEATURES
# =============================================================================

FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]

X      = x_train[FEATURES].values
y      = y_train["p0q0"].values
X_test = x_test[FEATURES].values

# =============================================================================
# 3. ENTRAÎNEMENT SUR 100% DU TRAIN
# =============================================================================

print("Entraînement sur 100% des données...")

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X, y)

# =============================================================================
# 4. PRÉDICTION ET SOUMISSION
# =============================================================================

y_pred = rf.predict(X_test)

# Arrondi à l'entier (la cible est en minutes entières)
y_pred_rounded = np.round(y_pred).astype(int)

submission = pd.DataFrame({
    "index": x_test["Unnamed: 0"],
    "p0q0":  y_pred_rounded
})

submission.to_csv("submission.csv", index=False)
print(f"submission.csv généré ({len(submission):,} lignes)")

Entraînement sur 100% des données...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   46.3s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.2min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.7s finished


submission.csv généré (20,657 lignes)


In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

x_train = pd.read_csv(
    "x_train_final.csv",
    header=None,
    names=["idx", "idx2", "train", "gare", "date", "arret",
           "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"],
    skiprows=1
)
y_train = pd.read_csv("y_train_final_j5KGWWK.csv")

print(f"x_train : {x_train.shape[0]:,} lignes")

# =============================================================================
# 2. FEATURES
# =============================================================================

x_train["date"] = pd.to_datetime(x_train["date"])
jours_ohe = pd.get_dummies(x_train["date"].dt.dayofweek, prefix="jour")
 
FEATURES_NUM = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
 
X = pd.concat([x_train[FEATURES_NUM], jours_ohe], axis=1).values
y = y_train["p0q0"].values

# =============================================================================
# 3. SPLIT 90 / 10
# =============================================================================

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

print(f"Train      : {X_tr.shape[0]:,} lignes")
print(f"Validation : {X_val.shape[0]:,} lignes")

# =============================================================================
# 4. RANDOM FOREST
# =============================================================================

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

# =============================================================================
# 5. ÉVALUATION
# =============================================================================

y_pred = rf.predict(X_val)

print(f"\nMAE  : {mean_absolute_error(y_val, y_pred):.4f}")

x_train : 667,264 lignes
Train      : 600,537 lignes
Validation : 66,727 lignes


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   51.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.4min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.9s



MAE  : 0.8926


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.4s finished


In [9]:
# Tendance du train (retards des trains précédents)
x_train["mean_retard_train"] = x_train[["p2q0","p3q0","p4q0"]].mean(axis=1)

# Tendance de la gare (retards aux gares précédentes)
x_train["mean_retard_gare"] = x_train[["p0q2","p0q3","p0q4"]].mean(axis=1)

# Retard total "contexte global" (moyenne de toutes les valeurs passées)
x_train["mean_retard_global"] = x_train[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].mean(axis=1)

In [10]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(X_val)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.9min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.0s


MAE : 0.8653


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    3.0s finished


In [11]:
# Écart-type des retards (volatilité : est-ce que le train est régulièrement en retard ou c'est chaotique ?)
x_train["std_retard_train"]  = x_train[["p2q0","p3q0","p4q0"]].std(axis=1)
x_train["std_retard_gare"]   = x_train[["p0q2","p0q3","p0q4"]].std(axis=1)

# Tendance : est-ce que le retard s'aggrave ou s'améliore ?
x_train["tendance_train"] = x_train["p2q0"] - x_train["p4q0"]  # retard récent - retard ancien
x_train["tendance_gare"]  = x_train["p0q2"] - x_train["p0q4"]

# Retard maximum observé (cas extrêmes)
x_train["max_retard"] = x_train[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].max(axis=1)
x_train["min_retard"] = x_train[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].min(axis=1)

In [12]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(X_val)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  4.3min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.3s


MAE : 0.8604


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    3.1s finished


In [13]:
from sklearn.preprocessing import LabelEncoder

le_train = LabelEncoder()
le_gare  = LabelEncoder()

x_train["train_enc"] = le_train.fit_transform(x_train["train"])
x_train["gare_enc"]  = le_gare.fit_transform(x_train["gare"])

In [14]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(X_val)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  5.1min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.1s


MAE : 0.7864


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    3.0s finished


In [15]:
stats_gare = x_train.join(y_train[["p0q0"]]).groupby("gare")["p0q0"].agg(
    gare_mean="mean",
    gare_std="std",
    gare_median="median",
    gare_q25=lambda x: x.quantile(0.25),
    gare_q75=lambda x: x.quantile(0.75)
).reset_index()

x_train = x_train.merge(stats_gare, on="gare", how="left")

In [16]:
stats_train = x_train.join(y_train[["p0q0"]]).groupby("train")["p0q0"].agg(
    train_mean="mean",
    train_std="std",
    train_median="median"
).reset_index()

x_train = x_train.merge(stats_train, on="train", how="left")

In [17]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc",
            "gare_mean", "gare_std", "gare_median", "gare_q25", "gare_q75",
            "train_mean", "train_std", "train_median"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(X_val)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  3.0min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  8.1min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.6s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    3.9s finished


MAE : 0.6713


In [18]:
# On repart de x_train propre (sans les colonnes stats déjà ajoutées)
cols_a_supprimer = ["gare_mean","gare_std","gare_median","gare_q25","gare_q75",
                    "train_mean","train_std","train_median"]
x_train = x_train.drop(columns=cols_a_supprimer)

# Split d'abord
X_idx = np.arange(len(x_train))
idx_tr, idx_val = train_test_split(X_idx, test_size=0.10, random_state=42)

train_fold = x_train.iloc[idx_tr].copy()
val_fold   = x_train.iloc[idx_val].copy()
y_tr       = y_train["p0q0"].iloc[idx_tr].values
y_val      = y_train["p0q0"].iloc[idx_val].values

# Stats calculées UNIQUEMENT sur les 90%
stats_gare = train_fold.join(y_train["p0q0"].iloc[idx_tr].rename("p0q0")).groupby("gare")["p0q0"].agg(
    gare_mean="mean", gare_std="std", gare_median="median",
    gare_q25=lambda x: x.quantile(0.25), gare_q75=lambda x: x.quantile(0.75)
).reset_index()

stats_train = train_fold.join(y_train["p0q0"].iloc[idx_tr].rename("p0q0")).groupby("train")["p0q0"].agg(
    train_mean="mean", train_std="std", train_median="median"
).reset_index()

# On applique sur train ET val (le val utilise les stats du train → pas de fuite)
train_fold = train_fold.merge(stats_gare, on="gare", how="left")
train_fold = train_fold.merge(stats_train, on="train", how="left")

val_fold = val_fold.merge(stats_gare, on="gare", how="left")
val_fold = val_fold.merge(stats_train, on="train", how="left")

In [19]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc",
            "gare_mean", "gare_std", "gare_median", "gare_q25", "gare_q75",
            "train_mean", "train_std", "train_median"]

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(train_fold[FEATURES].values, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(val_fold[FEATURES].values)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  3.2min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  9.1min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.5s


MAE : 0.7518


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:   10.2s finished


In [ ]:
import networkx as nx

G = nx.DiGraph()

for (train, date), groupe in x_train.groupby(["train", "date"]):
    groupe = groupe.sort_values("arret")
    gares  = groupe["gare"].tolist()

    for i in range(len(gares)):
        G.add_node(gares[i])

        if i > 0:
            G.add_edge(gares[i-1], gares[i]) # ajouter retard et q0p0, trouver retard moyen entre deux gares

print(f"Noeuds (gares) : {G.number_of_nodes()}")
print(f"Arêtes (liaisons) : {G.number_of_edges()}")

Noeuds (gares) : 84
Arêtes (liaisons) : 709


In [23]:
# Rang relatif de la gare dans le trajet (0 = départ, 1 = terminus)
position_gare = {}
for (train, date), groupe in x_train.groupby(["train", "date"]):
    groupe = groupe.sort_values("arret")
    n = len(groupe)
    for i, (_, row) in enumerate(groupe.iterrows()):
        position_gare[row["gare"]] = position_gare.get(row["gare"], [])
        position_gare[row["gare"]].append(i / (n - 1) if n > 1 else 0)

position_df = pd.DataFrame({
    "gare": list(position_gare.keys()),
    "position_moyenne": [np.mean(v) for v in position_gare.values()]
})

train_fold = train_fold.merge(position_df, on="gare", how="left")
val_fold   = val_fold.merge(position_df, on="gare", how="left")

In [24]:
freq_gare = x_train.groupby(["gare", "date"])["train"].nunique().groupby("gare").mean().reset_index()
freq_gare.columns = ["gare", "freq_trains_par_jour"]

train_fold = train_fold.merge(freq_gare, on="gare", how="left")
val_fold   = val_fold.merge(freq_gare, on="gare", how="left")

In [25]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc",
            "gare_mean", "gare_std", "gare_median", "gare_q25", "gare_q75",
            "train_mean", "train_std", "train_median",
            "position_moyenne", "freq_trains_par_jour"]

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(train_fold[FEATURES].values, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(val_fold[FEATURES].values)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  3.4min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  9.4min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    3.0s finished


MAE : 0.7505


In [ ]:
# Fusion temporaire pour avoir la date et le target ensemble
train_with_target = train_fold.copy()
train_with_target["p0q0"] = y_tr
train_with_target["date"] = pd.to_datetime(train_with_target["date"])

def stats_7j(df, group_col):
    records = []
    for date in df["date"].unique():
        date_min = date - pd.Timedelta(days=7)
        fenetre  = df[(df["date"] > date_min) & (df["date"] < date)]
        stats    = fenetre.groupby(group_col)["p0q0"].agg(
            **{f"{group_col}_mean_7j": "mean",
               f"{group_col}_std_7j":  "std"}
        ).reset_index()
        stats["date"] = date
        records.append(stats)
    return pd.concat(records, ignore_index=True)

stats_train_7j = stats_7j(train_with_target, "train")
stats_gare_7j  = stats_7j(train_with_target, "gare")

# Merge sur train_fold et val_fold
train_fold["date"] = pd.to_datetime(train_fold["date"])
val_fold["date"]   = pd.to_datetime(val_fold["date"])

train_fold = train_fold.merge(stats_train_7j, on=["train", "date"], how="left")
train_fold = train_fold.merge(stats_gare_7j,  on=["gare",  "date"], how="left")

val_fold = val_fold.merge(stats_train_7j, on=["train", "date"], how="left")
val_fold = val_fold.merge(stats_gare_7j,  on=["gare",  "date"], how="left")

# Valeurs manquantes (train/gare sans historique sur 7j) remplacées par la moyenne globale
train_fold["train_mean_7j"] = train_fold["train_mean_7j"].fillna(train_fold["train_mean"])
train_fold["train_std_7j"]  = train_fold["train_std_7j"].fillna(train_fold["train_std"])
train_fold["gare_mean_7j"]  = train_fold["gare_mean_7j"].fillna(train_fold["gare_mean"])
train_fold["gare_std_7j"]   = train_fold["gare_std_7j"].fillna(train_fold["gare_std"])

val_fold["train_mean_7j"] = val_fold["train_mean_7j"].fillna(val_fold["train_mean"])
val_fold["train_std_7j"]  = val_fold["train_std_7j"].fillna(val_fold["train_std"])
val_fold["gare_mean_7j"]  = val_fold["gare_mean_7j"].fillna(val_fold["gare_mean"])
val_fold["gare_std_7j"]   = val_fold["gare_std_7j"].fillna(val_fold["gare_std"])

In [29]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc",
            "gare_mean", "gare_std",
            "train_mean", "train_std", "train_median",
            "position_moyenne", "freq_trains_par_jour",
            "train_mean_7j", "train_std_7j",
            "gare_mean_7j",  "gare_std_7j"]

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(train_fold[FEATURES].values, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(val_fold[FEATURES].values)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  4.4min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 11.4min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    3.4s finished


MAE : 0.7467


In [ ]:
x_test = pd.read_csv("x_test_final.csv")
x_test["date"] = pd.to_datetime(x_test["date"])

# Features de base
x_test["mean_retard_train"]  = x_test[["p2q0","p3q0","p4q0"]].mean(axis=1)
x_test["mean_retard_gare"]   = x_test[["p0q2","p0q3","p0q4"]].mean(axis=1)
x_test["mean_retard_global"] = x_test[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].mean(axis=1)
x_test["std_retard_train"]   = x_test[["p2q0","p3q0","p4q0"]].std(axis=1)
x_test["std_retard_gare"]    = x_test[["p0q2","p0q3","p0q4"]].std(axis=1)
x_test["tendance_train"]     = x_test["p2q0"] - x_test["p4q0"]
x_test["tendance_gare"]      = x_test["p0q2"] - x_test["p0q4"]
x_test["max_retard"]         = x_test[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].max(axis=1)
x_test["min_retard"]         = x_test[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].min(axis=1)

# Encodage train & gare
# Encodage robuste aux valeurs inconnues
train_mapping = {v: i for i, v in enumerate(le_train.classes_)}
gare_mapping  = {v: i for i, v in enumerate(le_gare.classes_)}

x_test["train_enc"] = x_test["train"].map(train_mapping).fillna(-1).astype(int)
x_test["gare_enc"]  = x_test["gare"].map(gare_mapping).fillna(-1).astype(int)

# Stats globales (calculées sur train_fold)
x_test = x_test.merge(stats_gare,  on="gare",  how="left")
x_test = x_test.merge(stats_train, on="train", how="left")

# Position & fréquence
x_test = x_test.merge(position_df, on="gare", how="left")
x_test = x_test.merge(freq_gare,   on="gare", how="left")

# Stats 7 jours
x_test = x_test.merge(stats_train_7j, on=["train", "date"], how="left")
x_test = x_test.merge(stats_gare_7j,  on=["gare",  "date"], how="left")

# Fallback valeurs manquantes
x_test["train_mean_7j"] = x_test["train_mean_7j"].fillna(x_test["train_mean"])
x_test["train_std_7j"]  = x_test["train_std_7j"].fillna(x_test["train_std"])
x_test["gare_mean_7j"]  = x_test["gare_mean_7j"].fillna(x_test["gare_mean"])
x_test["gare_std_7j"]   = x_test["gare_std_7j"].fillna(x_test["gare_std"])

# Prédiction
y_pred = rf.predict(x_test[FEATURES].values)
y_pred_rounded = np.round(y_pred).astype(int)

submission = pd.DataFrame({
    "index": x_test["Unnamed: 0"],
    "p0q0":  y_pred_rounded
})

submission.to_csv("submission2+.csv", index=False)
print(f"submission+.csv généré ({len(submission):,} lignes)")

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s


submission2.csv généré (20,657 lignes)


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.3s finished


In [33]:
importances = pd.Series(rf.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=False)
print(importances.to_string())

gare_mean_7j            0.143452
train_mean_7j           0.085343
train_mean              0.083368
train_std_7j            0.058034
train_std               0.057798
arret                   0.048648
gare_std_7j             0.044015
mean_retard_gare        0.041541
train_enc               0.039016
std_retard_train        0.034812
gare_mean               0.032185
mean_retard_train       0.027735
p0q2                    0.023894
gare_enc                0.023777
std_retard_gare         0.023762
mean_retard_global      0.022215
gare_std                0.020690
tendance_gare           0.018973
p0q3                    0.018518
p2q0                    0.018427
p0q4                    0.017887
position_moyenne        0.017860
train_median            0.016643
freq_trains_par_jour    0.016065
p3q0                    0.015694
tendance_train          0.014856
p4q0                    0.013708
min_retard              0.012760
max_retard              0.008323
